# PatchTST Analysis -- ETTh1

Post-training analysis for PatchTST on ETTh1. Run after attaching the training output dataset.

**Prerequisites**
- Training notebook completed successfully
- Training outputs attached as a dataset at `/kaggle/input/patchtst-outputs/`
- ETTh1.csv attached at `/kaggle/input/etth1-dataset/ETTh1.csv`

**Outputs** (written to `/kaggle/working/plots/`)
- `forecast_vs_truth_pred96.png` -- 3 representative test windows, OT channel
- `val_mse_curves.png` -- validation MSE across all four horizons
- `per_channel_mse_pred96.png` -- per-variate test MSE bar chart

In [ ]:
# Cell 1 -- Setup
# Run this notebook after attaching the training outputs as a Kaggle dataset,
# or locally by updating RESULTS_DIR and DATA_PATH to the downloaded paths.
import matplotlib
matplotlib.use("Agg")  # headless backend: no display required
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader

# Adjust these paths when running locally.
RESULTS_DIR = Path("/kaggle/input/patchtst-outputs/results")
DATA_PATH   = Path("/kaggle/input/etth1-dataset/ETTh1.csv")
PLOTS_DIR   = Path("/kaggle/working/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

VARIATE_NAMES = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]
SEQ_LEN       = 512
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Results dir : {RESULTS_DIR}")
print(f"Data path   : {DATA_PATH}")
print(f"Plots dir   : {PLOTS_DIR}")
print(f"Device      : {DEVICE}")

In [ ]:
# Cell 2 -- ETTh1Dataset
from __future__ import annotations

from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

_TRAIN_END: int = 8640
_VAL_END:   int = 11520
_TEST_END:  int = 14400

Split = Literal["train", "val", "test"]


class ETTh1Dataset(Dataset):
    """Sliding window dataset over the ETTh1 time series.

    Split protocol matches PatchTST, iTransformer, and TimeMixer papers:
        Train : rows [0,     8640)
        Val   : rows [8640,  11520)
        Test  : rows [11520, 14400)

    Normalisation: per-channel z-score, scaler fitted on train split only.

    Args:
        csv_path: Path to ETTh1.csv.
        split:    One of 'train', 'val', 'test'.
        seq_len:  Number of input timesteps.
        pred_len: Number of target timesteps immediately following the input.
    """

    def __init__(self, csv_path: str | Path, split: Split, seq_len: int, pred_len: int) -> None:
        super().__init__()
        if split not in ("train", "val", "test"):
            raise ValueError(f"split must be one of 'train', 'val', 'test', got '{split}'")
        if seq_len < 1:
            raise ValueError(f"seq_len must be >= 1, got {seq_len}")
        if pred_len < 1:
            raise ValueError(f"pred_len must be >= 1, got {pred_len}")

        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.split    = split

        raw = self._load_csv(Path(csv_path))
        self._validate_length(raw)

        train_rows = raw[:_TRAIN_END]
        self._mean = train_rows.mean(axis=0)
        self._std  = train_rows.std(axis=0, ddof=0).clip(min=1e-8)
        normalized = (raw - self._mean) / self._std

        start, end    = self._split_bounds(split)
        self._data    = normalized[start:end].astype(np.float32)

        window = seq_len + pred_len
        if len(self._data) < window:
            raise ValueError(
                f"Split '{split}' has {len(self._data)} rows but "
                f"seq_len + pred_len = {window}. Reduce seq_len or pred_len."
            )
        self._num_samples = len(self._data) - window + 1

    def __len__(self) -> int:
        return self._num_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        if idx < 0 or idx >= self._num_samples:
            raise IndexError(f"Index {idx} out of range [0, {self._num_samples})")
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)

    @property
    def num_features(self) -> int:
        """Number of channels / variates."""
        return self._data.shape[1]

    @staticmethod
    def _load_csv(path: Path) -> np.ndarray:
        if not path.exists():
            raise FileNotFoundError(
                f"ETTh1.csv not found at {path}. "
                "Download from https://github.com/zhouhaoyi/ETDataset"
            )
        df = pd.read_csv(path)
        numeric = df.drop(columns=["date"])
        if numeric.isnull().any().any():
            raise ValueError("ETTh1.csv contains NaN values; preprocessing required.")
        return numeric.to_numpy(dtype=np.float64)

    @staticmethod
    def _validate_length(data: np.ndarray) -> None:
        if len(data) < _TEST_END:
            raise ValueError(
                f"ETTh1.csv has {len(data)} rows but at least {_TEST_END} are required "
                "for the standard 12/4/4 month split."
            )

    @staticmethod
    def _split_bounds(split: Split) -> tuple[int, int]:
        bounds: dict[str, tuple[int, int]] = {
            "train": (0,          _TRAIN_END),
            "val":   (_TRAIN_END, _VAL_END),
            "test":  (_VAL_END,   _TEST_END),
        }
        return bounds[split]


# Smoke test
_ds = ETTh1Dataset.__new__(ETTh1Dataset)
print("ETTh1Dataset defined.")

In [ ]:
# Cell 3 -- PatchTST model
# Reference: Nie et al., "A Time Series Is Worth 64 Words", ICLR 2023.
# https://arxiv.org/abs/2211.14730

import math
import torch
import torch.nn as nn


class PatchEmbedding(nn.Module):
    """Extract overlapping patches from a univariate series and project to d_model.

    Args:
        patch_size: Number of time steps per patch.
        stride:     Stride between consecutive patches.
        d_model:    Projection dimension.
        dropout:    Dropout rate applied after embedding.
    """

    def __init__(self, patch_size: int, stride: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.projection = nn.Linear(patch_size, d_model)
        self.dropout    = nn.Dropout(dropout)

    @staticmethod
    def _sinusoidal_encoding(num_patches: int, d_model: int, device: torch.device) -> torch.Tensor:
        position = torch.arange(num_patches, dtype=torch.float, device=device).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float, device=device)
            * (-math.log(10000.0) / d_model)
        )
        encoding = torch.zeros(1, num_patches, d_model, device=device)
        encoding[0, :, 0::2] = torch.sin(position * div_term)
        encoding[0, :, 1::2] = torch.cos(position * div_term[: d_model // 2])
        return encoding

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, seq_len, 1)
        Returns:
            (B, num_patches, d_model)
        """
        x       = x.squeeze(-1)
        pad_len = self.stride - ((x.size(1) - self.patch_size) % self.stride)
        if pad_len < self.stride:
            x = torch.nn.functional.pad(x, (0, pad_len))
        x = x.unfold(dimension=1, size=self.patch_size, step=self.stride)
        x = self.projection(x)
        x = x + self._sinusoidal_encoding(x.size(1), x.size(2), x.device)
        return self.dropout(x)


class _TransformerBlock(nn.Module):
    """Pre-norm transformer block: LayerNorm -> attention -> residual,
    LayerNorm -> MLP -> residual."""

    def __init__(self, d_model: int, num_heads: int, mlp_ratio: int = 4, dropout: float = 0.0) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp   = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * mlp_ratio, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed      = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x           = x + attn_out
        x           = x + self.mlp(self.norm2(x))
        return x


class TransformerEncoder(nn.Module):
    """Stack of pre-norm transformer blocks with a final layer norm.

    Args:
        d_model:    Model dimension.
        num_heads:  Number of attention heads.
        num_layers: Number of stacked transformer blocks.
        dropout:    Dropout rate.
    """

    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float) -> None:
        super().__init__()
        self.blocks = nn.ModuleList(
            [_TransformerBlock(d_model, num_heads, dropout=dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block in self.blocks:
            x = block(x)
        return self.norm(x)


class ForecastHead(nn.Module):
    """Flatten patch tokens and project to a forecast horizon.

    Args:
        num_patches: Number of patch tokens.
        d_model:     Model dimension.
        pred_len:    Forecast horizon.
    """

    def __init__(self, num_patches: int, d_model: int, pred_len: int) -> None:
        super().__init__()
        self.flatten = nn.Flatten(start_dim=1)
        self.linear  = nn.Linear(num_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(self.flatten(x))


class PatchTST(nn.Module):
    """Channel-independent PatchTST for multivariate long-horizon forecasting.

    Channels are processed independently: the input is reshaped from (B, seq_len, C)
    to (B*C, seq_len, 1) before the encoder. Weights are shared across channels by
    construction. This is the PatchTST/64 configuration from Nie et al., ICLR 2023.

    Args:
        seq_len:      Input sequence length.
        pred_len:     Forecast horizon.
        num_variates: Number of input channels.
        patch_size:   Time steps per patch. Default 16 (paper config).
        stride:       Stride between patches. Default 8 (paper config).
        d_model:      Transformer model dimension. Default 128 (paper config).
        num_heads:    Number of attention heads. Default 16 (paper config).
        num_layers:   Number of transformer blocks. Default 3 (paper config).
        dropout:      Dropout rate. Default 0.2 (paper config).
    """

    def __init__(
        self,
        seq_len:      int,
        pred_len:     int,
        num_variates: int,
        patch_size:   int   = 16,
        stride:       int   = 8,
        d_model:      int   = 128,
        num_heads:    int   = 16,
        num_layers:   int   = 3,
        dropout:      float = 0.2,
    ) -> None:
        super().__init__()
        self.num_variates = num_variates
        self.pred_len     = pred_len

        pad_len     = stride - ((seq_len - patch_size) % stride)
        padded_len  = seq_len + (pad_len if pad_len < stride else 0)
        num_patches = (padded_len - patch_size) // stride + 1

        self.patch_embedding = PatchEmbedding(patch_size, stride, d_model, dropout)
        self.encoder         = TransformerEncoder(d_model, num_heads, num_layers, dropout)
        self.head            = ForecastHead(num_patches, d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, seq_len, C)
        Returns:
            (B, pred_len, C)
        """
        B, seq_len, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, seq_len, 1)
        x = self.patch_embedding(x)
        x = self.encoder(x)
        x = self.head(x)
        x = x.reshape(B, C, self.pred_len).permute(0, 2, 1)
        return x


# Shape smoke test
_x   = torch.randn(2, 512, 7)
_out = PatchTST(seq_len=512, pred_len=96, num_variates=7)(_x)
assert _out.shape == (2, 96, 7), f"Unexpected shape: {_out.shape}"
del _x, _out
print("PatchTST defined and shape verified.")

In [ ]:
# Cell 4 -- Plot 1: Forecast vs ground truth (3 representative test windows, pred_len=96)
PRED_LEN        = 96
CHECKPOINT_PATH = RESULTS_DIR / "checkpoints" / f"patchtst_pred{PRED_LEN}_best.pt"

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}\n"
        "Attach the training output dataset before running this notebook."
    )

test_ds      = ETTh1Dataset(DATA_PATH, split="test", seq_len=SEQ_LEN, pred_len=PRED_LEN)
num_variates = test_ds.num_features

model = PatchTST(seq_len=SEQ_LEN, pred_len=PRED_LEN, num_variates=num_variates)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

n       = len(test_ds)
indices = [0, n // 2, n - 1]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
with torch.no_grad():
    for ax, idx in zip(axes, indices):
        x, y  = test_ds[idx]
        pred  = model(x.unsqueeze(0).to(DEVICE)).squeeze(0).cpu().numpy()
        truth = y.numpy()

        ax.plot(truth[:, 6], label="Ground truth (OT)", color="steelblue",  linewidth=1.5)
        ax.plot(pred[:, 6],  label="Forecast (OT)",     color="darkorange", linewidth=1.5, linestyle="--")
        ax.set_title(f"Test window index {idx}  (position {idx} of {n - 1})")
        ax.set_xlabel("Steps ahead")
        ax.set_ylabel("Normalised value")
        ax.legend(fontsize=9)

fig.suptitle(
    f"PatchTST | pred_len={PRED_LEN} | OT channel forecast vs ground truth",
    fontsize=13,
)
plt.tight_layout()
out = PLOTS_DIR / f"forecast_vs_truth_pred{PRED_LEN}.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

In [ ]:
# Cell 5 -- Plot 2: Validation MSE learning curves for all four horizons
fig, ax = plt.subplots(figsize=(11, 5))

for pred_len in [96, 192, 336, 720]:
    csv_path = RESULTS_DIR / f"patchtst_pred{pred_len}.csv"
    if not csv_path.exists():
        print(f"Missing: {csv_path} -- skipping pred_len={pred_len}.")
        continue
    df = pd.read_csv(csv_path)
    ax.plot(df["epoch"], df["val_mse"], label=f"pred_len={pred_len}")

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation MSE")
ax.set_title("PatchTST validation MSE across forecast horizons (ETTh1)")
ax.legend()
plt.tight_layout()
out = PLOTS_DIR / "val_mse_curves.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

In [ ]:
# Cell 6 -- Plot 3: Per-channel test MSE (pred_len=96)
# Per-channel MSE is averaged over batch and time dimensions per batch, then
# divided by the number of batches -- a consistent relative ranking metric.
# OT (oil temperature, index 6) is typically the hardest channel to forecast.
PRED_LEN        = 96
CHECKPOINT_PATH = RESULTS_DIR / "checkpoints" / f"patchtst_pred{PRED_LEN}_best.pt"

test_ds = ETTh1Dataset(DATA_PATH, split="test", seq_len=SEQ_LEN, pred_len=PRED_LEN)
model   = PatchTST(seq_len=SEQ_LEN, pred_len=PRED_LEN, num_variates=test_ds.num_features)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

loader          = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
channel_mse_sum = np.zeros(len(VARIATE_NAMES))
n_batches       = 0

with torch.no_grad():
    for x, y in loader:
        pred             = model(x.to(DEVICE)).cpu().numpy()
        truth            = y.numpy()
        channel_mse_sum += ((pred - truth) ** 2).mean(axis=(0, 1))
        n_batches       += 1

per_channel_mse = channel_mse_sum / n_batches

print(f"Per-channel test MSE (pred_len={PRED_LEN}):")
for name, mse in zip(VARIATE_NAMES, per_channel_mse):
    print(f"  {name}: {mse:.4f}")
hardest = VARIATE_NAMES[int(np.argmax(per_channel_mse))]
print(f"\nHardest channel: {hardest}")

fig, ax = plt.subplots(figsize=(9, 4))
bars    = ax.bar(VARIATE_NAMES, per_channel_mse, color="steelblue", edgecolor="white")
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
ax.set_xlabel("Variate")
ax.set_ylabel("Test MSE")
ax.set_title(f"PatchTST per-channel test MSE | pred_len={PRED_LEN} | ETTh1")
plt.tight_layout()
out = PLOTS_DIR / f"per_channel_mse_pred{PRED_LEN}.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

In [ ]:
# Cell 7 -- Final summary table vs published benchmarks
published = {
    96:  (0.370, 0.400),
    192: (0.413, 0.422),
    336: (0.422, 0.440),
    720: (0.447, 0.468),
}

summary_path = RESULTS_DIR / "patchtst_ettch1.csv"
if not summary_path.exists():
    raise FileNotFoundError(f"Summary CSV not found: {summary_path}")

df = pd.read_csv(summary_path)

print(f"{'pred_len':>10} {'MSE (ours)':>12} {'MAE (ours)':>12} {'MSE (paper)':>13} {'MAE (paper)':>13} {'MSE gap':>10}")
print("-" * 75)
for _, row in df.iterrows():
    pl           = int(row["pred_len"])
    mse          = row["test_mse"]
    mae          = row["test_mae"]
    p_mse, p_mae = published[pl]
    gap          = mse - p_mse
    print(
        f"{pl:>10} {mse:>12.4f} {mae:>12.4f} "
        f"{p_mse:>13.3f} {p_mae:>13.3f} {gap:>+10.4f}"
    )

print(f"\nAll plots saved to: {PLOTS_DIR}")